# Kaggle execution handoff

Template only; no Kaggle run or GPU time is claimed. Upload the approved corpus and code. Use a vLLM-format model approved as equivalent to the local pilot; MLX weights are not directly portable. Keep one model and retriever across all conditions. Do not spend paid API credit.


In [ ]:
import os
import subprocess
import time
from pathlib import Path

project = Path("/kaggle/working/lma-grounding-decay")
assert project.is_dir(), "Upload the project snapshot first"
os.chdir(project)
subprocess.run(["nvidia-smi"], check=True)


Install core dependencies and a GPU-compatible vLLM build appropriate to the Kaggle CUDA image. Record exact package versions in the compute log. Select an approved immutable model revision before starting the server. The local development revision is documented in `docs/reproduction.md`; backend format/quantisation equivalence still requires review.


In [ ]:
model_id = os.environ.get("LMA_APPROVED_MODEL_ID", "")
revision = os.environ.get("LMA_APPROVED_MODEL_REVISION", "")
assert model_id and revision, "Set the approved model identity and immutable revision"
with open("/kaggle/working/runtime_versions.txt", "w") as stream:
    subprocess.run(["python", "-m", "pip", "freeze"], stdout=stream, check=True)


In [ ]:
with open("/kaggle/working/vllm.log", "w") as stream:
    server = subprocess.Popen(["vllm", "serve", model_id, "--revision", revision, "--tensor-parallel-size", "2", "--dtype", "half", "--max-model-len", "12000"], stdout=stream, stderr=subprocess.STDOUT)


Wait until the server reports ready in its log. Import the frozen corpus and verify its manifest checksum. The command below runs the four primary development conditions; use the documented reranker environment for B1-R. Do not label unreviewed candidate queries as frozen gold.


In [ ]:
started = time.monotonic()
env = dict(os.environ, PYTHONPATH="src")
command = ["python", "-m", "conditions", "--queries", "data/eval/development_candidates.jsonl", "--conditions", "B0", "B1", "B2", "M", "--backend", "vllm", "--base-url", "http://127.0.0.1:8000/v1", "--model", model_id, "--revision", revision, "--output", "/kaggle/working/runs/development", "--resume"]
subprocess.run(command, env=env, check=True)
print({"phase_wall_seconds": time.monotonic()-started, "gpu_active_seconds": "not measured"})


In [ ]:
subprocess.run(["python", "-m", "eval.run_audit", "--runs", "/kaggle/working/runs/development", "--output", "/kaggle/working/integration_validation.json"], env=env, check=True)
server.terminate()


Download the run directory, audit, runtime versions and vLLM log before ending the session. Add actual session/GPU usage from Kaggle to the compute log; phase wall time is not GPU-active time.
